In [ ]:
import json

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pulp

## 1. 最適化の対象設定の読み込み

In [ ]:
with open("./data/conditions.json", "r", encoding="utf-8") as f:
    dict_condition = json.load(f)

In [ ]:
dict_condition

## 2. 線形計画問題（LP）

In [ ]:
# lpの最適化問題のインスタンスを定義

prob_lp = pulp.LpProblem("LP", pulp.LpMaximize)

In [ ]:
# _aをproduct_aの生産数、_bをproduct_bの生産数とする
# _tを東京、_fを福岡とする

# 東京のproduct_a, _bの生産数の変数を定義
lp_t_a = pulp.LpVariable("lp_t_a", lowBound=0, cat=pulp.LpInteger)
lp_t_b = pulp.LpVariable("lp_t_b", lowBound=0, cat=pulp.LpInteger)

# 福岡のproduct_a, _bの生産数の変数を定義
lp_f_a = pulp.LpVariable("lp_f_a", lowBound=0, cat=pulp.LpInteger)
lp_f_b = pulp.LpVariable("lp_f_b", lowBound=0, cat=pulp.LpInteger)

In [ ]:
# 目的関数の定義
# 東京と福岡を対象にする
# 東京のproduct_a+東京のproduct_b+福岡のproduct_a+福岡のproduct_b
# の利益を最大化する

prob_lp += (
    ( int(dict_condition["tokyo"]["product_a"]["sales_price"].split("_")[0]) 
        - int(dict_condition["tokyo"]["product_a"]["cost_price"].split("_")[0]) 
    ) * lp_t_a 
    + (int(dict_condition["tokyo"]["product_b"]["sales_price"].split("_")[0])
        - int(dict_condition["tokyo"]["product_b"]["cost_price"].split("_")[0])
    ) * lp_t_b 
    + (int(dict_condition["fukuoka"]["product_a"]["sales_price"].split("_")[0]) 
        - int(dict_condition["fukuoka"]["product_a"]["cost_price"].split("_")[0])
    ) * lp_f_a 
    + (int(dict_condition["fukuoka"]["product_b"]["sales_price"].split("_")[0]) 
        - int(dict_condition["fukuoka"]["product_b"]["cost_price"].split("_")[0])
    )* lp_f_b 
),"Objective"


In [ ]:
# 生産条件の追加

# 市場の需要の制限
prob_lp += lp_t_a <= int(dict_condition["tokyo"]["product_a"]["maximum_market_demand"].split("_")[0]) , "max_demand_ta"
prob_lp += lp_t_b <= int(dict_condition["tokyo"]["product_b"]["maximum_market_demand"].split("_")[0]) , "max_demand_tb"
prob_lp += lp_f_a <= int(dict_condition["fukuoka"]["product_a"]["maximum_market_demand"].split("_")[0]) , "max_demand_fa"
prob_lp += lp_f_b <= int(dict_condition["fukuoka"]["product_b"]["maximum_market_demand"].split("_")[0]) , "max_demand_fb"

# 工場の生産キャパシティーの制限
prob_lp += (
    lp_t_a * int(dict_condition["tokyo"]["product_a"]["production_time"].split("_")[0]) 
    +  lp_t_b * int(dict_condition["tokyo"]["product_b"]["production_time"].split("_")[0]) 
    <= int(dict_condition["tokyo"]["factory_capability"].split("_")[0]) 
), "factory_capacity_t"

prob_lp += (
    lp_f_a * int(dict_condition["fukuoka"]["product_a"]["production_time"].split("_")[0]) 
    +  lp_f_b * int(dict_condition["fukuoka"]["product_b"]["production_time"].split("_")[0]) 
    <= int(dict_condition["fukuoka"]["factory_capability"].split("_")[0]) 
), "factory_capacity_f"



In [ ]:
# 最適化の実行
status = prob_lp.solve()

In [ ]:
# LPの結果

print("Status:", pulp.LpStatus[status])
print("tokyo_product_a =", lp_t_a.varValue)
print("tokyo_product_b =", lp_t_b.varValue)
print("fukuoka_product_a =", lp_f_a.varValue)
print("fukuoka_product_b =", lp_f_b.varValue)

print("Objective Value =", pulp.value(prob_lp.objective))

## 3. 混合整数線形計画法(MILP)

In [ ]:
# milpの最適化問題のインスタンスを定義

prob_milp = pulp.LpProblem("MILP", pulp.LpMaximize)

In [ ]:
# tokyo, fukuoka, osakaを対象

# 東京のproduct_a, _bの生産数の変数を定義
milp_t_a = pulp.LpVariable("milp_t_a", lowBound=0, cat=pulp.LpInteger)
milp_t_b = pulp.LpVariable("milp_t_b", lowBound=0, cat=pulp.LpInteger)

# 福岡のproduct_a, _bの生産数の変数を定義
milp_f_a = pulp.LpVariable("milp_f_a", lowBound=0, cat=pulp.LpInteger)
milp_f_b = pulp.LpVariable("milp_f_b", lowBound=0, cat=pulp.LpInteger)

# 大阪のproduct_a, _bの生産数の変数を定義
milp_o_a = pulp.LpVariable("milp_o_a", lowBound=0, cat=pulp.LpInteger)
milp_o_b = pulp.LpVariable("milp_o_b", lowBound=0, cat=pulp.LpInteger)

In [ ]:
# 工場の稼働フラグの変数を定義

# 東京の稼働フラグ
milp_t_flag = pulp.LpVariable("milp_t_flag", cat=pulp.LpBinary)

# 福岡の稼働フラグ
milp_f_flag = pulp.LpVariable("milp_f_flag", cat=pulp.LpBinary)

# 大阪の稼働フラグ
milp_o_flag = pulp.LpVariable("milp_o_flag", cat=pulp.LpBinary)


In [ ]:
# 目的関数の定義
# の利益を最大化する

# 各拠点の利益（単価 × 生産量）を計算
tokyo_profit = (
    (int(dict_condition["tokyo"]["product_a"]["sales_price"].split("_")[0]) - int(dict_condition["tokyo"]["product_a"]["cost_price"].split("_")[0])) * milp_t_a 
    + (int(dict_condition["tokyo"]["product_b"]["sales_price"].split("_")[0]) - int(dict_condition["tokyo"]["product_b"]["cost_price"].split("_")[0])) * milp_t_b
)
fukuoka_profit = (
    (int(dict_condition["fukuoka"]["product_a"]["sales_price"].split("_")[0]) - int(dict_condition["fukuoka"]["product_a"]["cost_price"].split("_")[0])) * milp_f_a 
    + (int(dict_condition["fukuoka"]["product_b"]["sales_price"].split("_")[0]) - int(dict_condition["fukuoka"]["product_b"]["cost_price"].split("_")[0])) * milp_f_b
)
osaka_profit = (
    (int(dict_condition["osaka"]["product_a"]["sales_price"].split("_")[0]) - int(dict_condition["osaka"]["product_a"]["cost_price"].split("_")[0])) * milp_o_a  # ※元のコードがmilp_f_aになっていたので修正
    + (int(dict_condition["osaka"]["product_b"]["sales_price"].split("_")[0]) - int(dict_condition["osaka"]["product_b"]["cost_price"].split("_")[0])) * milp_o_b  # ※元のコードがmilp_f_bになっていたので修正
)

# 各拠点の固定費（稼働時のみ発生）
tokyo_cost = int(dict_condition["tokyo"]["factory_operation_cost"].split("_")[0]) * milp_t_flag
fukuoka_cost = int(dict_condition["fukuoka"]["factory_operation_cost"].split("_")[0]) * milp_f_flag
osaka_cost = int(dict_condition["osaka"]["factory_operation_cost"].split("_")[0]) * milp_o_flag

# 目的関数の設定
prob_milp += (tokyo_profit - tokyo_cost) + (fukuoka_profit - fukuoka_cost) + (osaka_profit - osaka_cost), "Objective"

In [ ]:
# 生産条件の追加

# 市場の需要の制限
prob_milp += milp_t_a <= int(dict_condition["tokyo"]["product_a"]["maximum_market_demand"].split("_")[0]) , "max_demand_ta"
prob_milp += milp_t_b <= int(dict_condition["tokyo"]["product_b"]["maximum_market_demand"].split("_")[0]) , "max_demand_tb"
prob_milp += milp_f_a <= int(dict_condition["fukuoka"]["product_a"]["maximum_market_demand"].split("_")[0]) , "max_demand_fa"
prob_milp += milp_f_b <= int(dict_condition["fukuoka"]["product_b"]["maximum_market_demand"].split("_")[0]) , "max_demand_fb"
prob_milp += milp_o_a <= int(dict_condition["osaka"]["product_a"]["maximum_market_demand"].split("_")[0]) , "max_demand_oa"
prob_milp += milp_o_b <= int(dict_condition["osaka"]["product_b"]["maximum_market_demand"].split("_")[0]) , "max_demand_ob"

# 工場の生産キャパシティーの制限
prob_milp += (
    milp_t_a * int(dict_condition["tokyo"]["product_a"]["production_time"].split("_")[0]) 
    +  milp_t_b * int(dict_condition["tokyo"]["product_b"]["production_time"].split("_")[0]) 
    <= int(dict_condition["tokyo"]["factory_capability"].split("_")[0]) * milp_t_flag 
), "factory_capacity_t"

prob_milp += (
    milp_f_a * int(dict_condition["fukuoka"]["product_a"]["production_time"].split("_")[0]) 
    +  milp_f_b * int(dict_condition["fukuoka"]["product_b"]["production_time"].split("_")[0]) 
    <= int(dict_condition["fukuoka"]["factory_capability"].split("_")[0]) * milp_f_flag
), "factory_capacity_f"

prob_milp += (
    milp_o_a * int(dict_condition["osaka"]["product_a"]["production_time"].split("_")[0]) 
    +  milp_o_b * int(dict_condition["osaka"]["product_b"]["production_time"].split("_")[0]) 
    <= int(dict_condition["osaka"]["factory_capability"].split("_")[0]) * milp_o_flag
), "factory_capacity_o"



In [ ]:
# 最適化の実行
status_milp = prob_milp.solve()

In [ ]:
# MILPの最適化の結果

print("Status:", pulp.LpStatus[status_milp])
print("tokyo_product_a =", milp_t_a.varValue)
print("tokyo_product_b =", milp_t_b.varValue)
print("fukuoka_product_a =", milp_f_a.varValue)
print("fukuoka_product_b =", milp_f_b.varValue)
print("osaka_product_a =", milp_o_a.varValue)
print("osaka_product_b =", milp_o_b.varValue)
print("tokyo_operation_flag:", milp_t_flag.varValue)
print("fukuoka_operation_flag:", milp_f_flag.varValue)
print("osaka_operation_flag:", milp_o_flag.varValue)

print("Objective Value =", pulp.value(prob_milp.objective))